# Entrenamiento de modelos — OCR E-14 (2ª vuelta)Banco de entrenamiento: armar datasets, definir arquitecturas, entrenar, mediry comparar corridas. Para *inspeccionar* actas y lecturas está el otro notebook,[`banco_pruebas_2v.ipynb`](banco_pruebas_2v.ipynb).Todo el bucle de entrenamiento está **escrito en el notebook**, no importado,para que puedas cambiar arquitectura, pérdida o augmentación y volver a correr.### Punto de partida| Modelo | % de actas que cuadran (tramo 12.000–13.500) ||---|---|| `digitnet.pt` (1ª vuelta, binario 28×28) | 17,5 % || gris 48×48 ronda 1 | 36,3 % || **`digitnet_2v_gris.pt` ronda 2** | **38,3 %** |### Tres cosas medidas que conviene no repetir1. **El color empeora** (42,7 % RGB vs 46,9 % gris, todo lo demás igual). `GRIS=True`.2. **El bootstrapping se agotó**: +2 puntos con 4,7× más datos.3. **Por qué se agotó** → el autoetiquetado por aritmética solo conserva actas   que cuadran, o sea donde el modelo *ya acertaba*. Es **ciego a su propio error   sistemático**, y por eso más rondas no arreglan nada.### El trabajo que sí tiene retorno hoy → sección 6El clasificador tiene 10 clases y **ninguna para el aspa (✱)** de anulación, asíque la lee como `7` con confianza 0,92–0,98 — el **16,3 %** de las casillas.La sección 6 arma las clases `aspa`/`guion`/`vacío`, con etiquetado a manoporque **la aritmética no puede generarlas**.Detalle en [`docs/OCR_2V.md`](../docs/OCR_2V.md).

## 0 · Configuración

In [ ]:
from pathlib import Path
import sys, json, time

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "e14").is_dir())
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "e14" / "extraccion"))

DATASET = RAIZ / "data/segunda_vuelta/digitos_2v_boot2.npz"   # ronda 2
CLAVEROS = RAIZ / "data/segunda_vuelta/e14_pdfs_claveros"
SALIDA   = RAIZ / "models"
REGISTRO = RAIZ / "data/segunda_vuelta/experimentos.jsonl"     # historial de corridas

GRIS   = True        # el color empeora — ver portada
DEV    = "cuda"
SEMILLA = 0

print("raíz:    ", RAIZ)
print("dataset: ", DATASET.name, "|", "existe" if DATASET.exists() else "NO EXISTE")
print("salida:  ", SALIDA)

## 1 · Cargar e inspeccionar el dataset**El split es por MESA, nunca por caja.** Los 27 dígitos de un acta compartenescáner, bolígrafo y persona: si caen a ambos lados del split, la validación midememorización y sale optimista.

In [ ]:
import numpy as np, cv2, torch, matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F

d = np.load(DATASET, allow_pickle=True)
X, y, meta = d["X"], d["y"], d["meta"]
mesas = np.array([m.split("|")[0] for m in meta])

print(f"cajas: {len(y):,}   mesas: {len(np.unique(mesas)):,}   forma: {X.shape}")
cuenta = np.bincount(y, minlength=10)
print(f"\ndistribución de clases:")
for c, n in enumerate(cuenta):
    print(f"  {c}: {n:>7,}  {'#' * int(60 * n / cuenta.max())}")
print(f"\ndesbalance: la clase 0 es {cuenta[0]/cuenta[1:].mean():.1f}x la media del resto")
print("(esperado: las casillas anuladas se autoetiquetan como 0 — ver sección 6)")

In [ ]:
def split_por_mesa(mesas, frac_val=0.2, semilla=SEMILLA):
    unicas = np.unique(mesas)
    rng = np.random.default_rng(semilla)
    rng.shuffle(unicas)
    val = set(unicas[:max(1, int(len(unicas) * frac_val))].tolist())
    es_val = np.array([m in val for m in mesas])
    return ~es_val, es_val


tr, va = split_por_mesa(mesas)
print(f"train: {tr.sum():,} cajas / {len(np.unique(mesas[tr])):,} mesas")
print(f"val:   {va.sum():,} cajas / {len(np.unique(mesas[va])):,} mesas")
assert not (set(mesas[tr]) & set(mesas[va])), "fuga de mesas entre train y val"
print("sin fuga de mesas entre particiones")

## 2 · ArquitecturaEditable: cambiá `Red` y volvé a correr desde acá. La de referencia son 3 bloques(32/64/128) con doble conv + BatchNorm y *global average pooling*.

In [ ]:
class Red(nn.Module):
    def __init__(self, n_clases=10, base=32, dropout=0.3, canales_ent=3):
        super().__init__()
        def bloque(ent, sal):
            return nn.Sequential(
                nn.Conv2d(ent, sal, 3, padding=1), nn.BatchNorm2d(sal), nn.ReLU(),
                nn.Conv2d(sal, sal, 3, padding=1), nn.BatchNorm2d(sal), nn.ReLU(),
                nn.MaxPool2d(2))
        self.c = nn.Sequential(bloque(canales_ent, base),
                               bloque(base, base * 2),
                               bloque(base * 2, base * 4))
        self.f = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                               nn.Dropout(dropout), nn.Linear(base * 4, n_clases))

    def forward(self, x):
        return self.f(self.c(x))


_r = Red()
print(_r)
print(f"\nparámetros entrenables: {sum(p.numel() for p in _r.parameters() if p.requires_grad):,}")

### 2.1 · AugmentaciónGeométrica suave + **fotométrica** (brillo/contraste): la iluminación del escánervaría mucho entre actas y el modelo no debe agarrarse de eso.

In [ ]:
def augmenta(x):
    """x: (B,C,H,W) en [0,1]."""
    B = x.shape[0]
    brillo = (torch.rand(B, 1, 1, 1, device=x.device) - 0.5) * 0.30
    contraste = 1.0 + (torch.rand(B, 1, 1, 1, device=x.device) - 0.5) * 0.40
    x = ((x - 0.5) * contraste + 0.5 + brillo).clamp(0, 1)

    ang = (torch.rand(B, device=x.device) - 0.5) * (2 * 8 * np.pi / 180)
    esc = 1.0 + (torch.rand(B, device=x.device) - 0.5) * 0.20
    tx = (torch.rand(B, device=x.device) - 0.5) * 0.16
    ty = (torch.rand(B, device=x.device) - 0.5) * 0.16
    cos, sin = torch.cos(ang) / esc, torch.sin(ang) / esc
    th = torch.zeros(B, 2, 3, device=x.device)
    th[:, 0, 0], th[:, 0, 1], th[:, 0, 2] = cos, -sin, tx
    th[:, 1, 0], th[:, 1, 1], th[:, 1, 2] = sin, cos, ty
    grid = F.affine_grid(th, x.shape, align_corners=False)
    return F.grid_sample(x, grid, padding_mode="border", align_corners=False)


# ver el efecto sobre 8 muestras
xs = torch.from_numpy(X[:8]).permute(0, 3, 1, 2).float().div(255)
fig, axs = plt.subplots(2, 8, figsize=(12, 3.2))
for i in range(8):
    axs[0, i].imshow(xs[i].permute(1, 2, 0).numpy()); axs[0, i].axis("off")
    axs[1, i].imshow(augmenta(xs)[i].permute(1, 2, 0).numpy()); axs[1, i].axis("off")
axs[0, 0].set_title("original", loc="left", fontsize=9)
axs[1, 0].set_title("augmentada", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

## 3 · EntrenarBucle explícito para que puedas tocarlo. Devuelve el historial para graficar.

In [ ]:
def a_gris_np(A):
    """Quita el color conservando 3 canales, en uint8 (no float): el control del
    experimento de color sin multiplicar por 4 la memoria."""
    g = np.stack([cv2.cvtColor(a, cv2.COLOR_BGR2GRAY) for a in A])
    return np.repeat(g[..., None], 3, axis=3)


def a_lote(A_uint8, idx, dev):
    """uint8 (N,H,W,3) -> float32 (B,3,H,W) en GPU, SOLO para las filas de idx.

    Deliberadamente NO se sube el dataset entero a la GPU. Cargarlo de una vez
    costaba ~3,6 GB de VRAM y ~5,5 GB de RAM con 131k cajas, y crecía lineal con
    el dataset (una ronda de 260k cajas pedía ~10,8 GB de RAM en una máquina de
    23 GB). Convertir por lote deja el pico en decenas de MB y hace que el
    tamaño del dataset ya no limite."""
    a = torch.from_numpy(A_uint8[idx]).to(dev, non_blocking=True)
    return a.permute(0, 3, 1, 2).float().div_(255)


def entrenar(X, y, tr, va, n_clases=10, epochs=25, batch=256, lr=2e-3,
             gris=GRIS, dev=DEV, semilla=SEMILLA, verbose=True):
    dev = dev if torch.cuda.is_available() else "cpu"
    torch.manual_seed(semilla)

    # se queda en RAM y en uint8; a GPU va sólo el lote de turno
    A = a_gris_np(X) if gris else X
    itr, iva = np.where(tr)[0], np.where(va)[0]
    ytr = torch.from_numpy(y[itr]).long().to(dev)
    yva = torch.from_numpy(y[iva]).long().to(dev)

    # Pesos de clase ACOTADOS. Sin el clip, una clase con 0-2 ejemplos (p.ej.
    # 'vacio' recién creada) recibe peso ~131.000 y el loss se va a 69: el
    # entrenamiento colapsa y la red predice siempre lo mismo. Las clases
    # ausentes van a peso 0 para que no aporten gradiente.
    cuenta = np.bincount(y[itr], minlength=n_clases).astype(np.float32)
    hay = cuenta > 0
    w = np.zeros(n_clases, np.float32)
    w[hay] = cuenta[hay].sum() / cuenta[hay]
    w[hay] = np.clip(w[hay] / w[hay].mean(), 0.2, 10.0)
    if (~hay).any():
        print(f"  (clases sin ejemplos, peso 0: {[i for i in range(n_clases) if not hay[i]]})")
    pesos = torch.from_numpy(w).to(dev)

    red = Red(n_clases=n_clases).to(dev)
    opt = torch.optim.AdamW(red.parameters(), lr=lr, weight_decay=1e-4)
    pasos = epochs * max(1, len(itr) // batch + 1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=pasos)
    lossf = nn.CrossEntropyLoss(weight=pesos, label_smoothing=0.05)

    def evaluar_val():
        red.eval(); aciertos = 0
        with torch.no_grad():
            for i in range(0, len(iva), 512):
                sl = slice(i, i + 512)
                aciertos += (red(a_lote(A, iva[sl], dev)).argmax(1) == yva[sl]).sum().item()
        return aciertos / max(1, len(iva))

    hist = {"loss": [], "val_acc": [], "lr": []}
    mejor, mejor_sd = 0.0, None
    t0 = time.time()
    for ep in range(1, epochs + 1):
        red.train()
        perm = np.random.default_rng(semilla + ep).permutation(len(itr))
        suma, n = 0.0, 0
        for i in range(0, len(perm), batch):
            idx = itr[perm[i:i + batch]]
            xb = augmenta(a_lote(A, idx, dev))
            yb = ytr[torch.from_numpy(perm[i:i + batch]).to(dev)]
            opt.zero_grad()
            loss = lossf(red(xb), yb)
            loss.backward(); opt.step(); sched.step()
            suma += loss.item() * len(idx); n += len(idx)
        acc = evaluar_val()
        hist["loss"].append(suma / n); hist["val_acc"].append(acc)
        hist["lr"].append(sched.get_last_lr()[0])
        if acc > mejor:
            mejor, mejor_sd = acc, {k: v.detach().cpu().clone() for k, v in red.state_dict().items()}
        if verbose and (ep % 5 == 0 or ep == 1):
            vram = torch.cuda.max_memory_allocated() / 1e9 if dev == "cuda" else 0
            print(f"  ep{ep:3d}  loss={suma/n:.4f}  val_acc={acc:.4f}  "
                  f"(mejor {mejor:.4f})  VRAM pico {vram:.2f} GB")
    hist["segundos"] = time.time() - t0
    hist["mejor_acc"] = mejor
    red.load_state_dict(mejor_sd)
    return red, hist

In [ ]:
EPOCHS = 25       # 25 para iterar rápido (~4 min); con 40 el modelo de referencia da 38,3 %

red, hist = entrenar(X, y, tr, va, n_clases=10, epochs=EPOCHS)
print(f"\nmejor val_acc: {hist['mejor_acc']:.4f}   en {hist['segundos']:.0f} s")
print("OJO: accuracy sobre etiquetas autogeneradas (sesgadas a lo fácil).")
print("La métrica que decide es el % de CUADRE — sección 4.")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.4))
axs[0].plot(hist["loss"]); axs[0].set_title("pérdida (train)"); axs[0].set_xlabel("época")
axs[1].plot(hist["val_acc"]); axs[1].set_title("accuracy por dígito (val)"); axs[1].set_xlabel("época")
axs[1].axhline(hist["mejor_acc"], ls="--", c="crimson", lw=1)
axs[2].plot(hist["lr"]); axs[2].set_title("learning rate"); axs[2].set_xlabel("época")
for a in axs: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 4 · La métrica que decide: % de CUADRELa accuracy por dígito está inflada porque las etiquetas salen del autoetiquetado.Lo que mide progreso real es releer actas **no vistas** y ver cuántas cuadran.⚠️ **Compará siempre sobre el mismo tramo.** Distintos departamentos tienendistinta calidad de escaneo: el mismo modelo da 46,9 % en un tramo y 36,3 % enotro. `DESDE` fijo o los números no significan nada.

In [ ]:
import posiciones_2v as P
from e14.comunes import CAND_2V
from e14.ocr.chequeo_aritmetico import chequear
from e14.ocr.dataset_color import cajas_de_acta

CASILLAS = [c[0] for c in P.CELDAS_2V]


def medir_cuadre(red, carpeta=CLAVEROS, desde=12000, n=300, gris=GRIS, dev=DEV,
                 interpretar=None):
    """% de actas que cuadran. `interpretar` mapea la salida cruda de la red a
    un valor de casilla (para modelos con clases extra — ver sección 6)."""
    dev = dev if torch.cuda.is_available() else "cpu"
    red = red.eval().to(dev)
    pdfs = [p for p in Path(carpeta).rglob("*.pdf") if "_logs" not in p.parts][desde:desde + n]
    ok = tot = degeneradas = 0
    detalle = []
    for p in pdfs:
        try:
            cajas = cajas_de_acta(p)
            if len(cajas) != 9:
                continue
            noms = [x for x in CASILLAS if x in cajas]
            plano = np.stack([c for x in noms for c in cajas[x]])
            if gris:
                plano = a_gris_np(plano)
            xb = torch.from_numpy(plano).permute(0, 3, 1, 2).float().div(255).to(dev)
            with torch.no_grad():
                pr = torch.softmax(red(xb), 1).cpu().numpy()
            cls, conf = pr.argmax(1), pr.max(1)
            vals = {}
            for i, nom in enumerate(noms):
                c3, f3 = cls[3*i:3*i+3], conf[3*i:3*i+3]
                vals[nom] = interpretar(c3) if interpretar else int("".join(map(str, c3)))
            # Un acta leída como TODO CEROS satisface la aritmética de forma
            # trivial (0+0+0 == 0 == 0) y el chequeo diría "cuadra". Un modelo
            # colapsado sacaría 100 % de cuadre siendo inútil — pasó de verdad
            # al entrenar 13 clases con pesos sin acotar. Una mesa sin votantes
            # en el E-11 no existe, así que se descarta como degenerada.
            if vals.get("TOTAL_E11", 0) == 0 and vals.get("SUMA_TOTAL", 0) == 0:
                degeneradas += 1; tot += 1; detalle.append(
                    {"pdf": p, "valores": vals, "cuadra": False, "degenerada": True})
                continue
            q = chequear(vals, cand=CAND_2V)
            cuadra = bool(q.cuadra_suma and q.cuadra_e11)
            ok += cuadra; tot += 1
            detalle.append({"pdf": p, "valores": vals, "cuadra": cuadra})
        except Exception:
            pass
    if degeneradas:
        print(f"  ! {degeneradas} actas leídas como todo-ceros (descartadas, "
              f"no cuentan como cuadre) — señal de modelo colapsado")
    return (ok / max(1, tot)), tot, detalle


TRAMO_DESDE, TRAMO_N = 12000, 300
pct, tot, detalle = medir_cuadre(red, desde=TRAMO_DESDE, n=TRAMO_N)
print(f"tramo {TRAMO_DESDE}..{TRAMO_DESDE+TRAMO_N}   actas leídas: {tot}")
print(f"CUADRAN: {pct:.1%}")
print(f"\nreferencias en ESTE mismo tramo:  digitnet 1ª vuelta 17,5 %  |  ronda 2 38,3 %")

## 5 · Registro de experimentosCada corrida se anexa a `experimentos.jsonl`. Sin esto es fácil perder la cuentade qué configuración dio qué, sobre todo porque la accuracy engaña y hay quemirar el cuadre.

In [ ]:
def registrar(nombre, hist, pct_cuadre, tramo, extra=None):
    fila = {"ts": time.strftime("%Y-%m-%d %H:%M:%S"), "nombre": nombre,
            "val_acc": round(hist["mejor_acc"], 4), "cuadre": round(pct_cuadre, 4),
            "tramo": tramo, "epochs": len(hist["loss"]),
            "segundos": round(hist["segundos"]), "gris": GRIS,
            "dataset": Path(DATASET).name, **(extra or {})}
    REGISTRO.parent.mkdir(parents=True, exist_ok=True)
    with open(REGISTRO, "a", encoding="utf-8") as f:
        f.write(json.dumps(fila, ensure_ascii=False) + "\n")
    return fila


def ver_registro():
    if not REGISTRO.exists():
        print("sin experimentos todavía"); return
    filas = [json.loads(l) for l in open(REGISTRO, encoding="utf-8")]
    print(f"{'fecha':20s} {'nombre':22s} {'val_acc':>8s} {'cuadre':>8s} {'tramo':>14s} {'ep':>4s}")
    print("-" * 82)
    for f in filas:
        print(f"{f['ts']:20s} {f['nombre'][:22]:22s} {f['val_acc']:>8.4f} "
              f"{f['cuadre']:>7.1%} {str(f.get('tramo','')):>14s} {f['epochs']:>4d}")


registrar(f"base_{EPOCHS}ep", hist, pct, f"{TRAMO_DESDE}+{TRAMO_N}")
ver_registro()

In [ ]:
# guardar el modelo si mejora lo que ya hay
GUARDAR_COMO = None       # p.ej. "digitnet_2v_v3.pt"

if GUARDAR_COMO:
    ruta = SALIDA / GUARDAR_COMO
    torch.save(red.state_dict(), ruta)
    print("guardado:", ruta)
else:
    print("poné GUARDAR_COMO = 'nombre.pt' para guardar")

## 6 · Extender las clases: `aspa`, `guion`, `vacío`**El trabajo con más retorno hoy.**El clasificador tiene 10 clases y ninguna para el **aspa (✱)** de anulación. Alverse forzado a elegir un dígito elige `7` sistemáticamente, con confianza0,92–0,98: una casilla `✱✱✱` que vale 0 se lee `777`. Pasa en el **16,3 %** delas casillas, concentrado en NULO / BLANCO / NO_MARCADO / TOTAL_INCINERADOS.**Estas etiquetas hay que hacerlas a mano.** El autoetiquetado por aritmética nosirve: solo conserva actas que cuadran, o sea donde el modelo ya acertaba, asíque nunca ve sus propios fallos. Son cientos de casillas, no miles, porque estánmuy concentradas.

In [ ]:
VOCAB = list("0123456789") + ["aspa", "guion", "vacio"]     # 13 clases
IDX = {v: i for i, v in enumerate(VOCAB)}
ATAJOS = {"a": "aspa", "g": "guion", "v": "vacio"}          # para teclear rápido
ETIQUETAS_CSV = RAIZ / "data/segunda_vuelta/etiquetas_simbolos.csv"

print("clases:", VOCAB)
print("atajos al etiquetar:", ATAJOS, "+ los dígitos 0-9 tal cual")

### 6.1 · Reunir candidatosSe buscan casillas donde el modelo actual lee `7` con confianza alta — ahí esdonde están las aspas. Es **aprendizaje activo**: etiquetar justo donde el modelose equivoca rinde mucho más que muestrear al azar.

In [ ]:
def reunir_candidatos(red, carpeta=CLAVEROS, desde=20000, n=150, gris=GRIS, dev=DEV):
    """Devuelve cajas sueltas (48x48) sospechosas de ser aspas, con su contexto."""
    dev = dev if torch.cuda.is_available() else "cpu"
    red = red.eval().to(dev)
    pdfs = [p for p in Path(carpeta).rglob("*.pdf") if "_logs" not in p.parts][desde:desde + n]
    cands = []
    for p in pdfs:
        try:
            cajas = cajas_de_acta(p)
            if len(cajas) != 9:
                continue
            noms = [x for x in CASILLAS if x in cajas]
            plano = np.stack([c for x in noms for c in cajas[x]])
            arr = a_gris_np(plano) if gris else plano
            xb = torch.from_numpy(arr).permute(0, 3, 1, 2).float().div(255).to(dev)
            with torch.no_grad():
                pr = torch.softmax(red(xb), 1).cpu().numpy()
            cls, conf = pr.argmax(1), pr.max(1)
            for i, nom in enumerate(noms):
                for j in range(3):
                    k = 3 * i + j
                    if cls[k] == 7 and conf[k] > 0.9:
                        cands.append({"caja": plano[k], "pdf": p, "casilla": nom,
                                      "pos": j, "conf": float(conf[k])})
        except Exception:
            pass
    return cands


cands = reunir_candidatos(red, desde=20000, n=150)
print(f"candidatos a aspa reunidos: {len(cands)}")
from collections import Counter
print(Counter(c["casilla"] for c in cands).most_common())

### 6.2 · EtiquetarSe muestran numeradas. Rellená `MIS_ETIQUETAS` con `índice: valor` usando losatajos (`a` aspa, `g` guion, `v` vacío) o el dígito real si de verdad lo es.Lo que no pongas se ignora — podés hacerlo por tandas.

In [ ]:
TANDA = 40
inicio = 0                      # subilo para seguir con la siguiente tanda

lote = cands[inicio:inicio + TANDA]
cols = 10
filas = int(np.ceil(len(lote) / cols))
fig, axs = plt.subplots(filas, cols, figsize=(cols * 1.25, filas * 1.45))
axs = np.atleast_2d(axs)
for i, ax in enumerate(axs.ravel()):
    ax.set_xticks([]); ax.set_yticks([])
    if i < len(lote):
        ax.imshow(cv2.cvtColor(lote[i]["caja"], cv2.COLOR_BGR2RGB))
        ax.set_title(f"{inicio + i}", fontsize=8)
    else:
        ax.axis("off")
plt.tight_layout(); plt.show()
print("Rellená MIS_ETIQUETAS en la celda siguiente con estos índices.")

In [ ]:
# índice -> etiqueta.  'a'=aspa  'g'=guion  'v'=vacio  '0'-'9'=dígito real
MIS_ETIQUETAS = {
    # 0: "a",
    # 1: "a",
    # 2: "7",
}


def guardar_etiquetas(dic, cands):
    import csv
    if not dic:
        print("MIS_ETIQUETAS está vacío — nada que guardar."); return 0
    nuevo = not ETIQUETAS_CSV.exists()
    ETIQUETAS_CSV.parent.mkdir(parents=True, exist_ok=True)
    with open(ETIQUETAS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["pdf", "casilla", "pos", "etiqueta"])
        for i, v in dic.items():
            v = ATAJOS.get(v, v)
            if v not in IDX:
                print(f"  ! índice {i}: '{v}' no es una clase válida, salteado"); continue
            c = cands[i]
            w.writerow([Path(c["pdf"]).resolve(), c["casilla"], c["pos"], v])
    print(f"guardadas {len(dic)} etiquetas -> {ETIQUETAS_CSV}")
    return len(dic)


guardar_etiquetas(MIS_ETIQUETAS, cands)

### 6.3 · Entrenar con 13 clasesMezcla los dígitos autoetiquetados (clases 0–9) con los símbolos etiquetados amano. Ojo: **el `0` autoetiquetado ya contiene aspas mal etiquetadas** (laaritmética las contó como 0 y acertó el total), así que ese ruido no desaparecesolo — a medida que juntes símbolos conviene reetiquetar los `0` dudosos.La regla de interpretación del E-14: todo-aspas → 0; un número a la derecha delaspa → ese número (`✱84` = 84). Es decir, aspa/guion/vacío **no aportan dígito**.

In [ ]:
def interpretar_simbolos(clases):
    """(3,) de índices de VOCAB -> valor entero de la casilla."""
    digs = [VOCAB[c] for c in clases]
    solo = [d for d in digs if d.isdigit()]
    return int("".join(solo)) if solo else 0


def cargar_dataset_extendido():
    """Une el npz autoetiquetado (0-9) con las etiquetas manuales de símbolos."""
    import csv
    Xs, ys, ms = [X], [y.astype(np.int64)], [mesas]
    if not ETIQUETAS_CSV.exists():
        print("todavía no hay etiquetas manuales — se entrena solo con 0-9")
        return np.concatenate(Xs), np.concatenate(ys), np.concatenate(ms), 10

    filas = list(csv.DictReader(open(ETIQUETAS_CSV, encoding="utf-8")))
    print(f"etiquetas manuales en el CSV: {len(filas)}")
    cajas_extra, y_extra, m_extra = [], [], []
    cache, fallos = {}, []
    for r in filas:
        p = Path(r["pdf"])
        if not p.is_absolute():          # CSV viejo con rutas relativas a la raíz
            p = (RAIZ / p).resolve()
        if p not in cache:
            try:
                cache[p] = cajas_de_acta(p)
            except Exception as e:
                cache[p] = None
                fallos.append(f"{p.name}: {e}")
        cj = cache[p]
        if not cj:
            fallos.append(f"{p.name}: no se pudo abrir"); continue
        if r["casilla"] not in cj:
            fallos.append(f"{p.name}: sin casilla {r['casilla']}"); continue
        cajas_extra.append(cj[r["casilla"]][int(r["pos"])])
        y_extra.append(IDX[r["etiqueta"]])
        m_extra.append("MANUAL_" + "_".join(P.parsear_clave(p)))
    if fallos:                            # nunca tragarse esto en silencio:
        print(f"  ! {len(fallos)} filas no se pudieron cargar. Primeras:")
        for f_ in fallos[:5]:
            print("   ", f_)
    if cajas_extra:
        Xs.append(np.stack(cajas_extra))
        ys.append(np.array(y_extra, np.int64))
        ms.append(np.array(m_extra))
        print(f"añadidas {len(cajas_extra)} cajas de símbolos")
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(ms), len(VOCAB)


Xe, ye, me, n_cls = cargar_dataset_extendido()
print(f"\ndataset: {len(ye):,} cajas · {n_cls} clases")
cnt = np.bincount(ye, minlength=n_cls)
print("por clase:", {VOCAB[i]: int(n) for i, n in enumerate(cnt) if n})
if n_cls > 10:
    print("símbolos:", {VOCAB[i]: int(cnt[i]) for i in range(10, n_cls)})

In [ ]:
# entrenar solo si de verdad hay clases nuevas; si no, no tiene sentido
MIN_POR_CLASE = 20      # por clase nueva, no en total

cnt_ext = np.bincount(ye, minlength=n_cls)
nuevas_ok = [i for i in range(10, n_cls) if cnt_ext[i] >= MIN_POR_CLASE]
nuevas_pocas = [VOCAB[i] for i in range(10, n_cls) if 0 < cnt_ext[i] < MIN_POR_CLASE]
nuevas_cero = [VOCAB[i] for i in range(10, n_cls) if cnt_ext[i] == 0]
if nuevas_pocas:
    print(f"clases con muy pocos ejemplos (<{MIN_POR_CLASE}): {nuevas_pocas} — "
          f"no alcanzan para aprenderlas, seguí etiquetando en 6.2")
if nuevas_cero:
    print(f"clases sin ningún ejemplo: {nuevas_cero}")

if n_cls > 10 and len(nuevas_ok) >= 1:
    tre, vae = split_por_mesa(me)
    red_ext, hist_ext = entrenar(Xe, ye, tre, vae, n_clases=n_cls, epochs=EPOCHS)
    pct_ext, tot_ext, _ = medir_cuadre(red_ext, desde=TRAMO_DESDE, n=TRAMO_N,
                                       interpretar=interpretar_simbolos)
    print(f"\nCUADRE con {n_cls} clases: {pct_ext:.1%}   (base 10 clases: {pct:.1%})")
    registrar(f"simbolos_{n_cls}cls", hist_ext, pct_ext, f"{TRAMO_DESDE}+{TRAMO_N}",
              extra={"n_manuales": int((ye >= 10).sum())})
    ver_registro()
else:
    print(f"Ninguna clase nueva llega a {MIN_POR_CLASE} ejemplos — no vale la pena entrenar.")
    print("Volvé a 6.2, etiquetá unas tandas más y corré esta celda de nuevo.")

## 7 · Bootstrapping (ciclo completo)Etiquetar con el mejor modelo → reentrenar → repetir.⚠️ **Ya se agotó**: +2 puntos con 4,7× más datos. Y se sabe por qué — el filtroaritmético solo deja pasar actas donde el modelo ya acertaba, así que el errorsistemático (las aspas) se auto-excluye del dataset en cada ronda. Queda acá parapoder repetirlo tras arreglar las clases, que es cuando debería volver a rendir.

In [ ]:
# se corre por CLI porque tarda (8 min por cada 12.000 actas)
print(f"""
# 1) re-etiquetar con el modelo actual
python -m e14.ocr.dataset_color construir {CLAVEROS} \\
    --salida datos_r3.npz --limite 24000 --dev cuda \\
    --modelo-48 models/digitnet_2v_gris.pt --gris

# 2) volver a este notebook con DATASET = datos_r3.npz y correr desde la sección 1
""")

---## Apéndice · qué mueve la aguja y qué no| Cambio | Efecto medido ||---|---|| Reentrenar sobre CLAVEROS + 48×48 sin binarizar | **17,5 % → 36,3 %** ✅ || Bootstrapping (4,7× más datos) | 36,3 % → 38,3 % ⚠️ agotado || Usar color RGB | 46,9 % → 42,7 % ❌ empeora || **Clases `aspa`/`guion`/`vacío`** | **sin medir — el 16,3 % de las casillas** |Y no perder de vista que **el techo no es 100 %**: un acta con error aritméticoreal —la denuncia #1 que el proyecto persigue— *nunca* debe cuadrar. Optimizar elcuadre a ciegas taparía justo lo que hay que detectar.